In [1]:
import pandas as pd

In [2]:
NIFTY500_URL = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

In [3]:
df = pd.read_csv(NIFTY500_URL)

In [4]:
df.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025


In [5]:
df.Industry.value_counts()

Industry
Financial Services                   101
Capital Goods                         63
Healthcare                            49
Automobile and Auto Components        37
Consumer Services                     29
Fast Moving Consumer Goods            28
Information Technology                27
Chemicals                             26
Metals & Mining                       18
Oil Gas & Consumable Fuels            18
Power                                 17
Consumer Durables                     16
Services                              14
Construction                          13
Construction Materials                11
Realty                                11
Telecommunication                     10
Textiles                               5
Media Entertainment & Publication      4
Diversified                            3
Name: count, dtype: int64

In [6]:
df[df["Series"] == "BE"]

,Company Name,Industry,Symbol,Series,ISIN Code
409,Schneider Electric Infrastructure Ltd.,Capital Goods,SCHNEIDER,BE,INE839M01018


In [7]:
df[df.Symbol == "ADANIENT"]

,Company Name,Industry,Symbol,Series,ISIN Code
16,Adani Enterprises Ltd.,Metals & Mining,ADANIENT,EQ,INE423A01024


In [8]:
# requests.get(MARKET_DATA_URL + "ADANIENT").json()
# Not working as plain requests are blocked by NSE. We need to use a session and set the user-agent and cookies to get the data.

In [9]:
import time
import json
import random
import requests
from pathlib import Path

In [10]:
NSE_BASE_URL = "https://www.nseindia.com"
MARKET_DATA_URL = "https://www.nseindia.com/api/quote-equity?symbol="
REFERER_URL = "https://www.nseindia.com/get-quotes/equity?symbol="

In [11]:
# Step 1: Create a session to persist cookies
session = requests.Session()

def get_market_data(session: requests.Session, symbol: str) -> dict:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": REFERER_URL + symbol
    }

    # Step 2: Make an initial request to get the cookies
    session.get(NSE_BASE_URL, headers=headers)
    # Step 3: Make the request to get market data
    response = session.get(MARKET_DATA_URL + symbol, headers=headers)
    
    return response.json()

get_market_data(session, "ADANIENT")

{'info': {'symbol': 'ADANIENT',
  'companyName': 'Adani Enterprises Limited',
  'industry': 'Trading - Minerals',
  'activeSeries': ['EQ', 'T0'],
  'debtSeries': [],
  'isFNOSec': True,
  'isCASec': False,
  'isSLBSec': True,
  'isDebtSec': True,
  'isSuspended': False,
  'tempSuspendedSeries': [],
  'isETFSec': False,
  'isDelisted': False,
  'isin': 'INE423A01024',
  'slb_isin': 'INE423A01024',
  'listingDate': '1997-06-04',
  'isMunicipalBond': False,
  'isHybridSymbol': False,
  'segment': 'EQUITY',
  'isTop10': False,
  'identifier': 'ADANIENTEQN'},
 'metadata': {'series': 'EQ',
  'symbol': 'ADANIENT',
  'isin': 'INE423A01024',
  'status': 'Listed',
  'listingDate': '04-Jun-1997',
  'industry': 'Trading - Minerals',
  'lastUpdateTime': '30-Mar-2026 16:00:00',
  'pdSectorPe': 70.02,
  'pdSymbolPe': 21.1,
  'pdSectorInd': 'NIFTY 50',
  'pdSectorIndAll': ['NIFTY 50',
   'NIFTY500 MULTICAP 50:25:25',
   'NIFTY100 ENHANCED ESG',
   'NIFTY50 EQUAL WEIGHT',
   'NIFTY 100',
   'NIFTY 200'

In [ ]:
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_nifty500_metadata(session: requests.Session, symbols: list[str]) -> None:
    for symbol in symbols[:10]:
        print(f"Fetching data for {symbol}...")
        data = get_market_data(session, symbol)
        time.sleep(random.uniform(1, 7))  # Sleep to avoid hitting rate limits
        # Save the data to a file
        output_file = output_dir / f"{symbol}.json"
        with open(output_file, "w") as f:
            json.dump(data, f, indent=4)

get_nifty500_metadata(session, df.Symbol.tolist())

Fetching data for 360ONE...
Fetching data for 3MINDIA...
Fetching data for ABB...
Fetching data for ACC...
Fetching data for ACMESOLAR...
Fetching data for AIAENG...
Fetching data for APLAPOLLO...
Fetching data for AUBANK...
Fetching data for AWL...
Fetching data for AADHARHFC...


In [12]:
# Should I have used playwright to fetch he data?
# Currently not needed as we are able to get the data using requests with proper headers and cookies.
# If below issues are faced, then we can consider using playwright.
# - If NSE blocks IP address after multiple requests
# - If data needs to be fetched from JS rendered pages not API endpoints

In [15]:
BASE_HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

def get_headers(symbol: str, referer_url: str) -> dict[str, str]:
    return {
        **BASE_HEADERS,
        "Referer": f"{referer_url}{symbol}",
    }

In [14]:
from typing import Optional

In [ ]:
# Retry with exonential backoff
def get_market_data_with_retry_v1(
        session: requests.Session, symbol: str, retries: int = 3
) -> Optional[dict]:
    for attempt in range(retries):
        try:
            headers = get_headers(symbol, REFERER_URL)
            session.get(NSE_BASE_URL, headers=headers)
            response = session.get(MARKET_DATA_URL + symbol, headers=headers)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 403:
                print(f"Access denied for {symbol}. Possible IP block. Attempt {attempt + 1} of {retries}.")
                session.cookies.clear()  # Clear cookies to try to bypass block
                session.get(NSE_BASE_URL, headers=headers)  # Get new cookies
            else:
                print(f"Unexpected status code {response.status_code} for {symbol}. Attempt {attempt + 1} of {retries}.")
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
        
        sleep_time = 2 ** attempt + random.uniform(0, 1)
        print(f"Retrying in {sleep_time:.2f} seconds...")
        time.sleep(sleep_time)
    
    raise Exception(f"Failed to fetch data for {symbol} after {retries} attempts")

get_market_data_with_retry_v1(session, "ADANIENT")

In [17]:
import asyncio